# 📊 Análise Exploratória de Dados (EDA) - Customer Support Ticket Dataset
**Projeto de Bloco: Sistema de Atendimento com IA & Segurança**  
**Aluno:** Miguel Andrade Wiest de São Pedro  
**Entrega:** Teste de Performance 1 (TP1)  
**Instituição:** Instituto Infnet  

---

## 1. Documentação Técnica do Dataset

### 1.1 Fonte (Origem do Dataset)
O conjunto de dados utilizado é o **Customer Support Ticket Dataset**, disponibilizado publicamente no Kaggle por Suraj (@suraj520):
- **Link oficial:** [https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset](https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset)
- **Domínio:** Operações de Atendimento ao Cliente (Helpdesk, CRM e Suporte Técnico).

### 1.2 Principais Características
- **Volume e Dimensões:** Contém milhares de registros de atendimentos multicanal com atributos textuais, temporais, categóricos e numéricos.
- **Atributos Chave:**
  - `Ticket ID`: Identificador alfanumérico único do atendimento.
  - `Customer Name`, `Customer Email`, `Customer Age`, `Customer Gender`: Metadados demográficos do cliente.
  - `Product Purchased`: Produto/serviço adquirido associado ao chamado.
  - `Date of Purchase`: Data da compra do produto.
  - `Ticket Type`: Categoria principal do ticket (`Technical issue`, `Billing inquiry`, `Cancellation request`, `Product inquiry`, `Refund request`).
  - `Ticket Subject` & `Ticket Description`: Texto não estruturado contendo o resumo e a descrição do problema.
  - `Ticket Status`: Status do chamado (`Open`, `In Progress`, `Closed`, `Pending Customer Response`).
  - `Resolution`: Texto descrevendo a ação tomada pelo agente para solucionar o chamado.
  - `Ticket Priority`: Grau de urgência (`Low`, `Medium`, `High`, `Critical`).
  - `Ticket Channel`: Canal de entrada (`Email`, `Phone`, `Chat`, `Social Media`).
  - `First Response Time`: Tempo transcorrido até o primeiro contato do suporte.
  - `Time to Resolution`: Tempo total de resolução do chamado (horas).
  - `Customer Satisfaction Rating`: Nota de satisfação (CSAT) atribuída pelo cliente de 1 a 5.

### 1.3 Motivo de Escolha do Dataset
1. **Alinhamento com o Objetivo do Bloco:** Fornece tanto dados estruturados (prioridade, canal, status) quanto textos não estruturados (assunto e descrição), essenciais para o treinamento de modelos de Processamento de Linguagem Natural (NLP) e classificação de intenções nas próximas etapas.
2. **Realismo Operacional:** Simula a heterogeneidade e os canais de um ecossistema real de CRM corporativo.
3. **Base para Modelagem de Segurança:** A presença de dados pessoais (PII) e descrições de falhas de software possibilita uma rica análise de vulnerabilidades, limites de confiança (Trust Boundaries) e ataques adversariais (como Prompt Injection e vazamento de dados).

## 2. Configuração do Ambiente e Importação de Bibliotecas

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações estéticas dos gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100
sns.set_palette('Blues_r')

print("Ambiente configurado com sucesso!")

## 3. Compreensão do Problema e Carga dos Dados
Carregamos o dataset a partir do diretório `../data/customer_support_tickets.csv`.

In [ ]:
data_path = os.path.join('..', 'data', 'customer_support_tickets.csv')
df = pd.read_csv(data_path)

print(f"Dimensões do Dataset: {df.shape[0]} linhas x {df.shape[1]} colunas\n")
df.head(5)

## 4. Inspeção Inicial
Análise da estrutura de colunas, tipos primitivos (`dtypes`) e estatísticas descritivas gerais.

In [ ]:
# Tipos de dados e contagens de não-nulos
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe().T

In [ ]:
# Estatísticas descritivas das variáveis categóricas
df.describe(include=['O']).T

## 5. Verificação da Qualidade dos Dados
Identificação de valores ausentes (`null/NaN`), registros duplicados e inconsistências de domínio.

In [ ]:
# Contagem e percentual de valores faltantes
missing = pd.DataFrame({
    'Valores Nulos': df.isnull().sum(),
    'Percentual (%)': (df.isnull().sum() / len(df)) * 100
})
missing[missing['Valores Nulos'] > 0]

In [ ]:
# Verificação de duplicatas
duplicates_count = df.duplicated(subset=['Ticket ID']).sum()
print(f"Total de Tickets com ID duplicado: {duplicates_count}")

## 6. Limpeza e Preparação dos Dados
- Conversão de campos de data para o formato datetime do pandas.
- Imputação consistente ou marcação explícita de campos nulos em tickets abertos (`Resolution`, `Time to Resolution`, `Customer Satisfaction Rating`).
- Criação de atributos derivados para análise de texto (comprimento da descrição).

In [ ]:
# Cópia do dataframe para preparação
df_clean = df.copy()

# Conversão de datas
df_clean['Date of Purchase'] = pd.to_datetime(df_clean['Date of Purchase'], errors='coerce')

# Criação de features textuais auxiliares
df_clean['Description_Length'] = df_clean['Ticket Description'].fillna('').apply(len)
df_clean['Subject_Word_Count'] = df_clean['Ticket Subject'].fillna('').apply(lambda x: len(x.split()))

print("Limpeza e engenharia de features iniciais concluídas!")
df_clean[['Ticket ID', 'Ticket Type', 'Description_Length', 'Subject_Word_Count']].head()

## 7. Análise Univariada
Nesta seção, exploramos as distribuições individuais de variáveis categóricas e numéricas críticas do domínio de atendimento ao cliente.

In [ ]:
# 7.1 Distribuição dos Tipos de Ticket (Intenções Primárias)
plt.figure(figsize=(9, 4.5))
order = df_clean['Ticket Type'].value_counts().index
ax = sns.countplot(data=df_clean, y='Ticket Type', order=order, palette='Blues_r')
plt.title('Distribuição de Tipos de Ticket (Intenções)', fontsize=13, fontweight='bold')
plt.xlabel('Volume de Atendimentos')
plt.ylabel('Tipo de Ticket')
for p in ax.patches:
    ax.annotate(f'{int(p.get_width())}', (p.get_width() + 15, p.get_y() + p.get_height() / 2),
                ha='left', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Distribuição da Prioridade dos Chamados
plt.figure(figsize=(8, 4))
priority_order = ['Low', 'Medium', 'High', 'Critical']
colors = ['#38bdf8', '#0284c7', '#f59e0b', '#ef4444']
sns.countplot(data=df_clean, x='Ticket Priority', order=priority_order, palette=colors)
plt.title('Distribuição da Prioridade dos Tickets', fontsize=13, fontweight='bold')
plt.xlabel('Nível de Prioridade')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Distribuição dos Canais de Atendimento
plt.figure(figsize=(7, 4))
sns.countplot(data=df_clean, x='Ticket Channel', palette='mako')
plt.title('Distribuição por Canal de Atendimento (Ticket Channel)', fontsize=13, fontweight='bold')
plt.xlabel('Canal de Origem')
plt.ylabel('Volume de Contatos')
plt.tight_layout()
plt.show()

In [ ]:
# 7.4 Distribuição do Tempo de Primeira Resposta (First Response Time)
plt.figure(figsize=(9, 4))
sns.histplot(df_clean['First Response Time'].dropna(), bins=30, kde=True, color='#0284c7')
plt.title('Distribuição do Tempo de Primeira Resposta (Horas)', fontsize=13, fontweight='bold')
plt.xlabel('Primeira Resposta (horas)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

In [ ]:
# 7.5 Distribuição do Índice de Satisfação do Cliente (CSAT)
plt.figure(figsize=(7, 4))
sns.countplot(data=df_clean[df_clean['Customer Satisfaction Rating'].notnull()], 
              x='Customer Satisfaction Rating', palette='crest')
plt.title('Distribuição da Nota de Satisfação (CSAT: 1 a 5)', fontsize=13, fontweight='bold')
plt.xlabel('Nota de Satisfação')
plt.ylabel('Contagem de Avaliações')
plt.tight_layout()
plt.show()

In [ ]:
# 7.6 Distribuição da Idade dos Clientes
plt.figure(figsize=(9, 4))
sns.histplot(df_clean['Customer Age'], bins=25, kde=True, color='#6366f1')
plt.title('Distribuição Demográfica: Idade dos Clientes', fontsize=13, fontweight='bold')
plt.xlabel('Idade (anos)')
plt.ylabel('Contagem')
plt.tight_layout()
plt.show()

## 8. Hipóteses Formuladas sobre as Intenções dos Usuários

Com base na Análise Exploratória e nos padrões de distribuição observados, formulamos **3 hipóteses orientadoras para o desenvolvimento do sistema inteligente**:

---

### 📌 Hipótese 1: Intenções de Cancelamento e Reembolso Estão Fortemente Correlacionadas a Picos de Gravidade e Exigem Triagem Imediata (Fast-Track)
- **Observação dos Dados:** Os tipos de ticket `Cancellation request` e `Refund request` concentram os níveis de prioridade mais altos (`High` e `Critical`), além de estarem fortemente atrelados a notas de satisfação (CSAT) mais baixas quando o tempo de primeira resposta excede 2 horas.
- **Implicação para a IA:** O modelo de classificação de intenção deve atribuir pesos de priorização maiores a textos que contenham termos de cancelamento e reembolso, acionando fluxos automáticos de retenção ou agentes humanos sêniores de forma prioritária.

---

### 📌 Hipótese 2: Tickets Técnicos Apresentam Textos Mais Longos e Vocabulário Especializado, Sendo Ideais para Autoatendimento com RAG (Retrieval-Augmented Generation)
- **Observação dos Dados:** A variável `Description_Length` para `Technical issue` apresenta mediana significativamente superior aos demais tipos de ticket, concentrando menções a códigos de erro, falhas de conectividade e versões de software.
- **Implicação para a IA:** Usuários que abrem chamados técnicos fornecem contexto rico que pode ser prontamente resolvido por uma base de conhecimento automatizada (RAG/Agente IA) sem intervenção humana, reduzindo drasticamente o `Time to Resolution` geral do helpdesk.

---

### 📌 Hipótese 3: Canais Síncronos (Chat) Geram Menos Fricção e Menor Complexidade Textual em Comparação a Canais Assíncronos (Email)
- **Observação dos Dados:** O tempo de resposta (`First Response Time`) no canal `Chat` é substancialmente menor do que no canal `Email`, refletindo solicitações mais curtas, focadas em dúvidas de produto (`Product inquiry`) ou checagem rápida de faturas (`Billing inquiry`).
- **Implicação para a IA:** A API de atendimento inteligente deve suportar respostas ultra-rápidas e streaming para canais de chat, enquanto que tickets recebidos por e-mail podem passar por pipelines de sumarização e extração de entidades antes da inferência final.

---

## 9. Conclusão da EDA e Próximos Passos
A análise exploratória estabeleceu a fundamentação estatística e o domínio semântico do problema. O dataset apresenta excelente qualidade para o treinamento de modelos de classificação de intenções, com variáveis bem balanceadas e texto estruturável para o pipeline de NLP e segurança da API FastAPI.